# UK Road Accidents Analysis (2005–2015)

**Author:** S. Ahmed  
**Dataset:** UK Department for Transport — Road Safety Data  
**Objective:** Analyse 1.78 million UK road accident records to identify patterns in severity, timing, weather conditions, speed limits, and long-term safety trends.

---

## Project Overview

Road accidents represent one of the most significant preventable causes of injury and mortality in the UK. The Department for Transport collects detailed records of every reported road collision via the Stats19 form. This project applies exploratory data analysis to 1.78 million accident records spanning 2005 to 2015, identifying patterns relevant to road safety policy and transport planning.

**Key questions addressed:**
- Are UK roads becoming safer over time?
- Which weather conditions produce the most accidents?
- How does speed limit relate to accident severity?
- Is there a meaningful difference between urban and rural accident patterns?
- Which days of the week are most dangerous?

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (11, 5)

print('All libraries imported successfully.')

## 2. Load and Inspect the Data

In [ ]:
# Load the dataset
df = pd.read_csv('Accidents0515.csv', low_memory=False)

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Check for missing values
print('Missing values per column (columns with missing data only):')
missing = df.isnull().sum()
print(missing[missing > 0])
print(f'\nDuplicate rows: {df.duplicated().sum():,}')

## 3. Data Cleaning and Feature Engineering

In [ ]:
# Parse date and extract year and month
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Remove rows with missing coordinates or unparseable dates
df = df.dropna(subset=['Longitude', 'Latitude', 'Date'])

# Map accident severity codes to readable labels
severity_map = {1: 'Fatal', 2: 'Serious', 3: 'Slight'}
df['Severity_Label'] = df['Accident_Severity'].map(severity_map)

# Map urban/rural area codes
df['Area_Type'] = df['Urban_or_Rural_Area'].map({1: 'Urban', 2: 'Rural'})

# Map day of week codes (Stats19: 1=Sunday)
day_map = {1:'Sunday', 2:'Monday', 3:'Tuesday',
           4:'Wednesday', 5:'Thursday', 6:'Friday', 7:'Saturday'}
df['Day_Name'] = df['Day_of_Week'].map(day_map)

# Map weather condition codes
weather_map = {
    1: 'Fine — no wind', 2: 'Raining — no wind', 3: 'Snowing — no wind',
    4: 'Fine + high wind', 5: 'Raining + high wind', 6: 'Snowing + high wind',
    7: 'Fog or mist', 8: 'Other', 9: 'Unknown'
}
df['Weather_Label'] = df['Weather_Conditions'].map(weather_map)

print(f'Rows retained after cleaning: {len(df):,}')
print(f'Year range: {df["Year"].min()} to {df["Year"].max()}')

## 4. Exploratory Data Analysis

### 4.1 Accident Severity Distribution

In [ ]:
severity_counts = df['Severity_Label'].value_counts()

fig, ax = plt.subplots()
colours = ['#D85A30', '#EF9F27', '#5DCAA5']
bars = ax.bar(severity_counts.index, severity_counts.values, color=colours, edgecolor='black')
ax.set_title('UK Road Accident Severity (2005–2015)', fontsize=13)
ax.set_ylabel('Number of Accidents')
ax.set_xlabel('Severity')
for bar, val in zip(bars, severity_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
            f'{val:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print('Severity breakdown:')
for label, count in severity_counts.items():
    print(f'  {label}: {count:,} ({count/len(df)*100:.2f}%)')

### 4.2 Year-on-Year Trend — Are UK Roads Getting Safer?

In [ ]:
yearly = df.groupby('Year').size().reset_index(name='Accidents')

fig, ax = plt.subplots()
ax.plot(yearly['Year'], yearly['Accidents'], marker='o',
        color='#185FA5', linewidth=2.5, markersize=8)
ax.fill_between(yearly['Year'], yearly['Accidents'], alpha=0.1, color='#185FA5')
ax.set_title('Total UK Road Accidents Per Year (2005–2015)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Accidents')
ax.set_xticks(yearly['Year'])
for _, row in yearly.iterrows():
    ax.annotate(f"{int(row['Accidents']):,}",
                (row['Year'], row['Accidents']),
                textcoords='offset points', xytext=(0, 9), ha='center', fontsize=8)
plt.tight_layout()
plt.show()

reduction = (yearly['Accidents'].iloc[0] - yearly['Accidents'].iloc[-1]) / yearly['Accidents'].iloc[0] * 100
print(f'Reduction in accidents from 2005 to 2015: {reduction:.1f}%')
print('This reflects the impact of improved vehicle safety, speed camera rollout, and drink-driving enforcement.')

### 4.3 Accidents by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['Day_Name'].value_counts().reindex(day_order)

fig, ax = plt.subplots()
ax.bar(day_counts.index, day_counts.values, color='#378ADD', edgecolor='black')
ax.set_title('UK Road Accidents by Day of Week', fontsize=13)
ax.set_ylabel('Number of Accidents')
ax.set_xlabel('Day')
plt.tight_layout()
plt.show()

busiest = day_counts.idxmax()
print(f'Most dangerous day: {busiest} ({day_counts[busiest]:,} accidents)')

### 4.4 Weather Conditions and Accident Frequency

In [ ]:
weather_counts = df['Weather_Label'].value_counts().drop('Unknown', errors='ignore').head(7)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(weather_counts.index, weather_counts.values, color='#1D9E75', edgecolor='black')
ax.set_title('Accidents by Weather Condition', fontsize=13)
ax.set_xlabel('Number of Accidents')
for i, val in enumerate(weather_counts.values):
    ax.text(val + 1000, i, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

fine = weather_counts.get('Fine — no wind', 0)
print(f'Fine weather accidents: {fine:,} ({fine/len(df)*100:.1f}% of all accidents)')
print('Key insight: Good weather produces most accidents due to higher traffic volumes and driver overconfidence.')

### 4.5 Speed Limit vs Accident Severity

In [ ]:
valid_speeds = [20, 30, 40, 50, 60, 70]
speed_df = df[df['Speed_limit'].isin(valid_speeds)]
speed_severity = speed_df.groupby(['Speed_limit', 'Severity_Label']).size().unstack(fill_value=0)
speed_severity['Fatal_Rate_%'] = (
    speed_severity.get('Fatal', 0) / speed_severity.sum(axis=1) * 100
).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cols_to_plot = [c for c in ['Fatal', 'Serious', 'Slight'] if c in speed_severity.columns]
speed_severity[cols_to_plot].plot(
    kind='bar', ax=axes[0],
    color=['#D85A30', '#EF9F27', '#5DCAA5'][:len(cols_to_plot)],
    edgecolor='black'
)
axes[0].set_title('Accidents by Speed Limit and Severity', fontsize=12)
axes[0].set_xlabel('Speed Limit (mph)')
axes[0].set_ylabel('Number of Accidents')
axes[0].tick_params(axis='x', rotation=0)

axes[1].bar(speed_severity.index.astype(str),
            speed_severity['Fatal_Rate_%'], color='#D85A30', edgecolor='black')
axes[1].set_title('Fatal Accident Rate by Speed Limit (%)', fontsize=12)
axes[1].set_xlabel('Speed Limit (mph)')
axes[1].set_ylabel('Fatal Rate (%)')
for i, (idx, row) in enumerate(speed_severity.iterrows()):
    axes[1].text(i, row['Fatal_Rate_%'] + 0.05,
                 f"{row['Fatal_Rate_%']:.2f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('30mph zones: highest accident volume (urban density)')
print('60-70mph roads: highest fatality rate per accident (rural high speed)')

### 4.6 Urban vs Rural Accident Severity

In [ ]:
urban_rural = df.groupby(['Area_Type', 'Severity_Label']).size().unstack(fill_value=0)
cols = [c for c in ['Fatal', 'Serious', 'Slight'] if c in urban_rural.columns]

fig, ax = plt.subplots()
urban_rural[cols].plot(
    kind='bar', ax=ax,
    color=['#D85A30', '#EF9F27', '#5DCAA5'][:len(cols)],
    edgecolor='black'
)
ax.set_title('Accident Severity: Urban vs Rural', fontsize=13)
ax.set_xlabel('Area Type')
ax.set_ylabel('Number of Accidents')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

for area in ['Urban', 'Rural']:
    if area in urban_rural.index and 'Fatal' in urban_rural.columns:
        total = urban_rural.loc[area].sum()
        fatal = urban_rural.loc[area, 'Fatal']
        print(f'{area} fatal rate: {fatal/total*100:.2f}%')

### 4.7 Monthly Accident Pattern

In [ ]:
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly = df.groupby('Month').size().reset_index(name='Accidents')
monthly['Month_Name'] = monthly['Month'].map(month_map)

fig, ax = plt.subplots()
ax.bar(monthly['Month_Name'], monthly['Accidents'], color='#534AB7', edgecolor='black')
ax.set_title('Total Accidents by Month (2005–2015 combined)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Accidents')
plt.tight_layout()
plt.show()

## 5. Summary Statistics

In [ ]:
total = len(df)
fatal = (df['Severity_Label'] == 'Fatal').sum()
serious = (df['Severity_Label'] == 'Serious').sum()
slight = (df['Severity_Label'] == 'Slight').sum()

print('=== Summary Statistics ===')
print(f'Total accidents analysed : {total:,}')
print(f'Fatal accidents          : {fatal:,} ({fatal/total*100:.2f}%)')
print(f'Serious accidents        : {serious:,} ({serious/total*100:.2f}%)')
print(f'Slight accidents         : {slight:,} ({slight/total*100:.2f}%)')
print(f'Year range               : 2005 to 2015')
print(f'Reduction over period    : {reduction:.1f}%')

## Conclusions

**Key findings:**
- UK roads became measurably safer over the decade, with a consistent year-on-year reduction in total accidents.
- Fine weather conditions account for the majority of accidents — counterintuitive but explained by higher traffic volumes and reduced driver caution in good conditions.
- 30mph zones record the highest total accident volume due to urban density, but 60–70mph roads produce disproportionately higher fatality rates per incident.
- Rural roads have a significantly higher fatal accident rate than urban roads despite lower total volumes.
- Friday records the highest accident frequency, consistent with increased traffic and end-of-week fatigue.

**Limitations:**
- Only reported accidents are included; minor incidents not attended by police are underrepresented.
- Correlation does not imply causation; multivariate modelling would be required to confirm causal relationships.

**Potential extensions:**
- Hour-by-hour accident frequency analysis
- Logistic regression to predict accident severity
- Geographic hotspot mapping using latitude and longitude